# Multilingual Document OCR + MRZ Parser
### Domino Data Lab · Arabic / French / English · Fully Offline · v6

---

## Complete bug history and fixes

| Version | Error | Root cause | Fix |
|---------|-------|-----------|-----|
| v1 | `AssertionError: param lang must in dict_keys, but got ar,fr,en` | PaddleOCR only accepts one language per instance | Two engines: `lang='ar'` + `lang='fr'` |
| v2–v3 | `ConnectionError` despite `det_model_dir` | PaddleOCR silently ignored the parameter | Copy models into `~/.paddleocr/whl/` cache |
| v4 | Models in wrong cache folder | `Path.home()` returned wrong path inside conda env | Read `BASE_DIR` from PaddleOCR source directly |
| v5 | Still downloading despite correct cache | `enable_hpi=True` needs `.onnx` files, only `.pdmodel` provided | **v6: `enable_hpi=False` + explicit model paths as double safety net** |

## v6 approach — three layers of defence
1. **Models in correct cache** (`/home/domino/.paddleocr/whl/...`) — checked and copied by Cell 2
2. **Explicit model paths** (`det_model_dir` + `rec_model_dir`) — points PaddleOCR directly to files
3. **Download blocker** — patches `requests.Session.send`; any download attempt shows a clear diagnostic error

## Before running
Complete Steps 1–3 in **INSTRUCTIONS.txt**:
1. **Laptop terminal** — download + extract the 3 model `.tar` files  
2. **Domino browser** — upload model folders to `/mnt/data/models/` and images to the documents folder  
3. **Domino terminal** — `pip install` the packages

---
## Cell 1 — Imports

In [ ]:
import json
import logging
import os
import re
import shutil
import time
from pathlib import Path
from typing import Any

import cv2
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from tqdm.notebook import tqdm

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger(__name__)

print('✓ Cell 1 — imports OK')

---
## Cell 2 — Configuration, Model Setup, Download Blocker

**Edit the two paths at the top** to match where your files are on Domino,  
then run the cell. Everything else is automatic.

In [ ]:
import requests as _requests

# =============================================================================
# ★ EDIT THESE TWO PATHS IF NEEDED ★
# =============================================================================

# Where your document images are (check what Domino shows you)
INPUT_DIR   = Path('/mnt/data/documents')   # change to /mnt/documents if needed

# Where you uploaded the extracted model folders
MODELS_DIR  = Path('/mnt/data/models')      # change to /mnt/models if needed

# =============================================================================
# OUTPUT PATHS (auto-created, no need to edit)
# =============================================================================

OUTPUT_JSON  = Path('/mnt/artifacts/results.json')
OUTPUT_EXCEL = Path('/mnt/artifacts/report.xlsx')

# =============================================================================
# PERFORMANCE
# =============================================================================

MAX_DIMENSION = 1280
CPU_THREADS   = os.cpu_count() or 4

# =============================================================================
# CONSTANTS
# =============================================================================

SUPPORTED_EXT    = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif', '.webp'}
MRZ_LINE_PATTERN = re.compile(r'^[A-Z0-9<]{30,}$')

# =============================================================================
# STEP A: Find PaddleOCR BASE_DIR from its own source — no guessing
# =============================================================================

try:
    from paddleocr.paddleocr import BASE_DIR as _PADDLE_BASE
    PADDLE_BASE = Path(_PADDLE_BASE)
    print(f'✓ PaddleOCR BASE_DIR (from source) : {PADDLE_BASE}')
except Exception as exc:
    import paddleocr as _poc
    PADDLE_BASE = Path(_poc.__file__).parent.parent.parent.parent / '.paddleocr'
    print(f'⚠ BASE_DIR fallback                : {PADDLE_BASE}  (reason: {exc})')

# =============================================================================
# STEP B: Model paths
#   - CACHE_*  : where PaddleOCR looks first (no download if these exist)
#   - SRC_*    : where you uploaded the extracted folders
# =============================================================================

# Cache paths — exact sub-folder structure PaddleOCR uses internally
CACHE_DET   = PADDLE_BASE / 'whl' / 'det' / 'multilingual' / 'Multilingual_PP-OCRv3_det_infer'
CACHE_AR    = PADDLE_BASE / 'whl' / 'rec' / 'arabic'        / 'arabic_PP-OCRv3_rec_infer'
CACHE_LATIN = PADDLE_BASE / 'whl' / 'rec' / 'latin'         / 'latin_PP-OCRv3_rec_infer'

# Source paths — your uploaded files
SRC_DET     = MODELS_DIR / 'Multilingual_PP-OCRv3_det_infer'
SRC_AR      = MODELS_DIR / 'arabic_PP-OCRv3_rec_infer'
SRC_LATIN   = MODELS_DIR / 'latin_PP-OCRv3_rec_infer'

print(f'  Cache det   : {CACHE_DET}')
print(f'  Cache ar    : {CACHE_AR}')
print(f'  Cache latin : {CACHE_LATIN}')

# =============================================================================
# STEP C: Copy models into cache (skip if already there)
# =============================================================================

def _is_valid_model_dir(p: Path) -> bool:
    """True only if inference.pdmodel exists inside the folder."""
    return p.is_dir() and (p / 'inference.pdmodel').exists()


def ensure_cached(src: Path, dst: Path, label: str) -> None:
    if _is_valid_model_dir(dst):
        print(f'  ✓ {label:35s} already in cache')
        return
    if not _is_valid_model_dir(src):
        raise FileNotFoundError(
            f'\n❌ Model not found or incomplete at: {src}\n'
            f'  Upload the folder to {MODELS_DIR}/ via Domino browser (Step 2 in INSTRUCTIONS.txt)\n'
            f'  The folder must contain: inference.pdmodel  inference.pdiparams  inference.pdiparams.info\n'
        )
    print(f'  → Copying {label} …')
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        shutil.rmtree(str(dst))
    shutil.copytree(str(src), str(dst))
    if not _is_valid_model_dir(dst):
        raise RuntimeError(f'Copy failed — inference.pdmodel not found at {dst}')
    print(f'  ✓ {label:35s} copied OK')


print('\nEnsuring models are in PaddleOCR cache …')
ensure_cached(SRC_DET,   CACHE_DET,   'Detection model')
ensure_cached(SRC_AR,    CACHE_AR,    'Arabic rec model')
ensure_cached(SRC_LATIN, CACHE_LATIN, 'Latin rec model (fr/en)')

# =============================================================================
# STEP D: Block ALL internet downloads — safety net
#
# WHY: Even with models in cache, if enable_hpi=True PaddleOCR looks for
#      .onnx files and tries to download them when not found.
# FIX: enable_hpi=False in _make_engine() + this blocker as insurance.
# =============================================================================

_orig_send = _requests.Session.send

def _no_paddle_downloads(self, request, **kwargs):
    url = getattr(request, 'url', str(request))
    if 'bcebos.com' in url:
        raise RuntimeError(
            f'\n\n❌ PaddleOCR tried to download from the internet!\n'
            f'   URL : {url}\n\n'
            f'   This file is missing from the cache.\n'
            f'   Run this in a Domino terminal to diagnose:\n'
            f'     ls {CACHE_DET}/\n'
            f'     ls {CACHE_AR}/\n'
            f'     ls {CACHE_LATIN}/\n'
            f'   Each folder must contain inference.pdmodel\n'
        )
    return _orig_send(self, request, **kwargs)

_requests.Session.send = _no_paddle_downloads

print('\n✓ Download blocker active')
print(f'\n✓ Cell 2 complete')
print(f'  Input dir    : {INPUT_DIR}')
print(f'  CPU threads  : {CPU_THREADS}')
print()
if not INPUT_DIR.exists():
    print(f'  ⚠ WARNING: {INPUT_DIR} does not exist!')
    print(f'     Check where your images are mounted in Domino.')
    print(f'     Common paths: /mnt/data/documents  or  /mnt/documents')
else:
    imgs = [p for p in INPUT_DIR.rglob('*') if p.suffix.lower() in SUPPORTED_EXT]
    print(f'  ✓ Found {len(imgs)} image(s) in {INPUT_DIR}')

---
## Cell 3 — Image Preprocessing

In [ ]:
def load_and_resize(image_path: str | Path, max_dim: int = MAX_DIMENSION) -> np.ndarray:
    """
    Load image and downscale if longest side > max_dim.
    cv2.INTER_AREA gives best quality when shrinking.
    """
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(f'Cannot read: {image_path}')
    h, w  = img.shape[:2]
    scale = min(max_dim / max(h, w), 1.0)
    if scale < 1.0:
        img = cv2.resize(img, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_AREA)
    return img


print('✓ Cell 3 — load_and_resize defined')

---
## Cell 4 — Build the Two OCR Engines

### Key fix vs v5: `enable_hpi=False`

| Setting | v5 (broken) | v6 (fixed) | Why |
|---------|------------|-----------|-----|
| `enable_hpi` | `True` | **`False`** | HPI needs `.onnx` files. We only have `.pdmodel`. With `True`, PaddleOCR downloaded ONNX models even though standard models were in cache. |
| `det_model_dir` | not set | **set explicitly** | Double safety net pointing directly at our cached files |
| `rec_model_dir` | not set | **set explicitly** | Same |

> ⏱ 10–30 seconds. You should see **no** `download` lines — only `✓`.

In [ ]:
from paddleocr import PaddleOCR


def _make_engine(lang: str, rec_model_dir: Path) -> PaddleOCR:
    """
    Create a single-language PaddleOCR engine — fully offline.

    Key settings
    ------------
    enable_hpi=False
        CRITICAL FIX. When True, PaddleOCR needs .onnx model files.
        Since we only have standard .pdmodel files, it tried to download
        ONNX versions from the internet even though .pdmodel was in cache.
        False = standard PaddlePaddle inference, works with our files.

    det_model_dir / rec_model_dir
        Explicit paths to our cached model folders.
        Double safety net on top of the cache placement in Cell 2.

    use_angle_cls=False
        Disables angle classification model entirely.
        Prevents a third model download for cls model.
        Documents are assumed upright.

    Parameters
    ----------
    lang          : 'ar' for Arabic, 'fr' for French + English
    rec_model_dir : path to the language-specific recognition model
    """
    return PaddleOCR(
        lang=lang,
        det_model_dir=str(CACHE_DET),       # explicit path — double safety net
        rec_model_dir=str(rec_model_dir),   # explicit path — double safety net
        use_angle_cls=False,                # no cls model needed
        use_gpu=False,                      # CPU-only
        enable_hpi=False,                   # ← KEY FIX: False avoids .onnx downloads
        cpu_threads=CPU_THREADS,
        show_log=False,
    )


# ── Arabic engine ─────────────────────────────────────────────────────────────
print('Initialising Arabic engine  (lang="ar", enable_hpi=False) …')
ocr_ar = _make_engine('ar', CACHE_AR)
print('✓ Arabic engine ready')

print()

# ── Latin engine ──────────────────────────────────────────────────────────────
print('Initialising Latin engine   (lang="fr", enable_hpi=False) …')
ocr_latin = _make_engine('fr', CACHE_LATIN)
print('✓ Latin engine ready  (French + English)')

print()
print('✓ Cell 4 complete — both engines ready, fully offline')

---
## Cell 5 — MRZ Detection and Parsing

In [ ]:
def detect_mrz_lines(text_lines: list[str]) -> list[str]:
    """
    Scan OCR lines for ICAO MRZ candidates.
    MRZ lines: only A-Z, 0-9, '<', length >= 30.
    Spaces removed first (OCR sometimes inserts them).
    """
    mrz = []
    for line in text_lines:
        cleaned = line.strip().replace(' ', '')
        if MRZ_LINE_PATTERN.match(cleaned):
            mrz.append(cleaned)
    return mrz


def parse_mrz_with_omnimrz(mrz_lines: list[str]) -> dict[str, Any]:
    """
    Parse + validate MRZ using OmniMRZ (ICAO 9303).
    Returns dict: raw_mrz, parsed_fields, validation, (optional) error.
    """
    try:
        from omnimrz import MRZ
    except ImportError:
        log.warning('omnimrz not installed — MRZ parsing skipped.')
        return {'error': 'omnimrz not installed', 'raw_mrz': mrz_lines}

    try:
        obj = MRZ('\n'.join(mrz_lines))
        fields = {
            k: getattr(obj, k, None) for k in [
                'document_type', 'issuing_country', 'document_number',
                'surname', 'given_names', 'nationality',
                'birth_date', 'sex', 'expiry_date',
                'optional_data_1', 'optional_data_2',
            ]
        }
        validation: dict[str, Any] = {}
        for attr in dir(obj):
            if 'valid' in attr.lower() or 'check' in attr.lower():
                try:
                    validation[attr] = getattr(obj, attr)
                except Exception:
                    pass
        overall = getattr(obj, 'valid', None)
        if overall is None:
            overall = all(v for v in validation.values() if isinstance(v, bool))
        validation['overall_valid'] = overall
        return {'raw_mrz': mrz_lines, 'parsed_fields': fields, 'validation': validation}
    except Exception as exc:
        log.warning('OmniMRZ failed: %s', exc)
        return {'raw_mrz': mrz_lines, 'error': str(exc)}


print('✓ Cell 5 — MRZ helpers defined')

---
## Cell 6 — Document Processor (dual engine)

In [ ]:
def _extract_lines(engine: Any, img: np.ndarray) -> list[str]:
    """Run one PaddleOCR engine, return flat list of text strings."""
    lines: list[str] = []
    raw = engine.ocr(img, cls=False)
    if raw and raw[0]:
        for item in raw[0]:
            if item and len(item) >= 2:
                txt = item[1][0] if isinstance(item[1], (list, tuple)) else str(item[1])
                lines.append(txt)
    return lines


def process_document(
    image_path:   str | Path,
    engine_ar:    Any,
    engine_latin: Any,
    max_dim:      int = MAX_DIMENSION,
) -> dict[str, Any]:
    """
    Run OCR with both engines, merge results, detect and parse MRZ.

    Steps:
      1. Load + resize image.
      2. engine_ar   → Arabic text lines.
      3. engine_latin → French/English text lines.
      4. Merge (Arabic first), remove exact duplicates.
      5. Scan for MRZ pattern.
      6. If >= 2 MRZ lines: parse with OmniMRZ.

    Returns dict: file_name, full_text, has_mrz, mrz_data, error, elapsed_sec
    """
    image_path = Path(image_path)
    result: dict[str, Any] = {
        'file_name':   image_path.name,
        'full_text':   '',
        'has_mrz':     False,
        'mrz_data':    None,
        'error':       None,
        'elapsed_sec': 0.0,
    }
    t0 = time.perf_counter()
    try:
        img         = load_and_resize(image_path, max_dim)
        lines_ar    = _extract_lines(engine_ar,    img)
        lines_latin = _extract_lines(engine_latin, img)

        seen: set[str] = set()
        text_lines: list[str] = []
        for line in lines_ar + lines_latin:
            if line not in seen:
                seen.add(line)
                text_lines.append(line)

        result['full_text'] = '\n'.join(text_lines)

        mrz_lines = detect_mrz_lines(text_lines)
        if len(mrz_lines) >= 2:
            result['has_mrz']  = True
            result['mrz_data'] = parse_mrz_with_omnimrz(mrz_lines)
            log.info('  ✓ MRZ detected (%d lines)', len(mrz_lines))
        else:
            log.info('  — No MRZ')

    except Exception as exc:
        log.error('Error %s: %s', image_path.name, exc)
        result['error'] = str(exc)

    result['elapsed_sec'] = round(time.perf_counter() - t0, 3)
    return result


print('✓ Cell 6 — process_document defined')

---
## Cell 7 — Test with One Sample Image
Change the path to one of your document images.

In [ ]:
SAMPLE_IMAGE = Path('/mnt/data/documents/sample.jpg')  # ← change to a real image

if SAMPLE_IMAGE.exists():
    print(f'Testing: {SAMPLE_IMAGE.name}')
    r = process_document(SAMPLE_IMAGE, ocr_ar, ocr_latin)
    print(f'\n  File       : {r["file_name"]}')
    print(f'  Has MRZ    : {r["has_mrz"]}')
    print(f'  Time       : {r["elapsed_sec"]}s')
    print(f'  Error      : {r["error"]}')
    print(f'\n--- OCR TEXT ---')
    print(r['full_text'] or '(nothing detected)')
    if r['has_mrz']:
        print('\n--- MRZ ---')
        print(json.dumps(r['mrz_data'], ensure_ascii=False, indent=2, default=str))
    print('\n✓ Cell 7 complete')
else:
    print(f'[SKIP] {SAMPLE_IMAGE} not found.')
    print('Change SAMPLE_IMAGE to a real path, or go to Cell 8.')

---
## Cell 8 — Batch Processing

In [ ]:
def collect_images(input_path: Path) -> list[Path]:
    """Return sorted list of images from a file or directory (recursive)."""
    if input_path.is_file():
        return [input_path]
    images = sorted(p for p in input_path.rglob('*') if p.suffix.lower() in SUPPORTED_EXT)
    if not images:
        print(f'⚠ No images found in {input_path}')
    return images


images = collect_images(INPUT_DIR)
print(f'Found {len(images)} image(s) in {INPUT_DIR}:')
for p in images:
    print(f'  {p.name}')

print()
all_results: list[dict[str, Any]] = []

for img_path in tqdm(images, desc='OCR + MRZ', unit='doc'):
    log.info('Processing: %s', img_path.name)
    r = process_document(img_path, ocr_ar, ocr_latin)
    all_results.append(r)
    log.info('  %.2fs | has_mrz=%s | error=%s', r['elapsed_sec'], r['has_mrz'], r['error'])

total = sum(r['elapsed_sec'] for r in all_results)
print(f'\n✓ Cell 8 — {len(all_results)} doc(s) done in {total:.1f}s')

---
## Cell 9 — Save JSON

In [ ]:
def save_json(results: list[dict[str, Any]], out: Path) -> None:
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, 'w', encoding='utf-8') as fh:
        json.dump(results, fh, ensure_ascii=False, indent=2, default=str)
    print(f'  {out.stat().st_size/1024:.1f} KB')


save_json(all_results, OUTPUT_JSON)
print(f'✓ Cell 9 — JSON → {OUTPUT_JSON}')

---
## Cell 10 — Save Excel (Summary · MRZ Details · Full Text)

In [ ]:
def _style_ws(ws, max_w: int = 60) -> None:
    fill = PatternFill(fill_type='solid', fgColor='BDD7EE')
    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.fill = fill
        cell.alignment = Alignment(horizontal='center', wrap_text=True)
    for col in ws.columns:
        letter = get_column_letter(col[0].column)
        best   = max((len(str(c.value or '')) for c in col), default=10)
        ws.column_dimensions[letter].width = min(best + 4, max_w)


def build_excel(results: list[dict[str, Any]], out: Path) -> None:
    out.parent.mkdir(parents=True, exist_ok=True)
    s_rows, m_rows, t_rows = [], [], []

    for r in results:
        md  = r.get('mrz_data') or {}
        val = md.get('validation', {}) if r.get('has_mrz') else {}
        ov  = val.get('overall_valid')
        s_rows.append({
            'file_name': r.get('file_name'), 'has_mrz': r.get('has_mrz'),
            'overall_mrz_valid': ov, 'elapsed_sec': r.get('elapsed_sec'),
            'error': r.get('error'),
        })
        t_rows.append({'file_name': r.get('file_name'), 'full_text': r.get('full_text')})
        if r.get('has_mrz') and md:
            pf = md.get('parsed_fields', {}) or {}
            m_rows.append({
                'file_name':       r.get('file_name'),
                'raw_mrz':         ' | '.join(md.get('raw_mrz', [])),
                'document_type':   pf.get('document_type'),
                'issuing_country': pf.get('issuing_country'),
                'document_number': pf.get('document_number'),
                'surname':         pf.get('surname'),
                'given_names':     pf.get('given_names'),
                'nationality':     pf.get('nationality'),
                'birth_date':      pf.get('birth_date'),
                'sex':             pf.get('sex'),
                'expiry_date':     pf.get('expiry_date'),
                'optional_data_1': pf.get('optional_data_1'),
                'optional_data_2': pf.get('optional_data_2'),
                'overall_valid':   ov,
                **{f'valid_{k}': v for k, v in val.items() if k != 'overall_valid'},
            })

    with pd.ExcelWriter(str(out), engine='openpyxl') as w:
        pd.DataFrame(s_rows).to_excel(w, sheet_name='Summary',     index=False)
        (pd.DataFrame(m_rows) if m_rows else pd.DataFrame(columns=['file_name'])
         ).to_excel(w, sheet_name='MRZ Details', index=False)
        pd.DataFrame(t_rows).to_excel(w, sheet_name='Full Text',   index=False)

    wb = load_workbook(str(out))
    for ws in wb.worksheets:
        _style_ws(ws)
    wb.save(str(out))
    print(f'  {out.stat().st_size/1024:.1f} KB')


build_excel(all_results, OUTPUT_EXCEL)
print(f'✓ Cell 10 — Excel → {OUTPUT_EXCEL}')

---
## Cell 11 — Final Summary

In [ ]:
df      = pd.read_excel(str(OUTPUT_EXCEL), sheet_name='Summary')
n       = len(df)
total_t = sum(r['elapsed_sec'] for r in all_results)

print('=' * 52)
print('  RESULTS SUMMARY')
print('=' * 52)
print(f'  Total documents    : {n}')
print(f'  With MRZ           : {int(df["has_mrz"].sum())}')
print(f'  MRZ valid          : {int(df["overall_mrz_valid"].sum())}')
print(f'  Errors             : {int(df["error"].notna().sum())}')
print(f'  Total time         : {total_t:.1f}s')
print(f'  Avg per document   : {total_t/max(n,1):.2f}s')
print('=' * 52)
print(f'  JSON  → {OUTPUT_JSON}')
print(f'  Excel → {OUTPUT_EXCEL}')
print('=' * 52)
print('Download from: Domino → Jobs → Artifacts')
df